In [12]:
l = list("A B C".encode("utf-8"))
l

[65, 32, 66, 32, 67]

In [13]:
word = bytes([65, 105, 95, 103]).decode("utf-8")
word

'Ai_g'

In [42]:
lis = [1,2,3,4,5]

for pair in zip(lis,lis[1:]):
    print(pair)
    

(1, 2)
(2, 3)
(3, 4)
(4, 5)


In [4]:
def logger(message:str):
    print("****"*15)
    print("message --> "+message)
    print("****"*15)

In [5]:
# ----------------
# Starting to implement a Tokenizer


In [6]:
with open("wiki.txt",'r') as f:
    content = f.read()

In [7]:
text_corpus = content
tokens = text_corpus.encode("utf-8") #-> these are raw bytes 
tokens = list(map(int,tokens)) # Covertining the encoded bytes tokens to int for data ops in later steps.

print("\n Inserted Text length -> :"+str(len(text_corpus)))
print("\n Produced bytes length -> :"+str(len(tokens)))



 Inserted Text length -> :178255102

 Produced bytes length -> :178723194


In [8]:
# Building a dictinary by counting number of consecutively occruing pairs.
# key in dict is the concecutive pairs
# value is the number of times it occured.
from typing import Any
def get_stats(tokens:list)->dict[tuple[int, int], int]:
    counts:dict[tuple[int, int], int] = {}
    for pair in zip(tokens,tokens[1:]):
        counts[pair] = counts.get(pair,0)+1
    return counts

stats  = get_stats(tokens)
# repositioned_stats = tuple((v,k) for k,v in stats.items()) # switched the places of key and values to easily sort them
                                                           # as below based on the first position of the item in the list so the dict can be 
                                                           # sorted on the number of counts
# sorted_stats = sorted(repositioned_stats,reverse=True)
# print(sorted_stats)
print("length of stats produced : -> "+str(len(stats)))

length of stats produced : -> 17484


In [9]:
# pairing Function , this takes input of the current tokens and pairs and new_index
# to be set for the new pair that need to be added to the dict 
# this will produce a new tokens list with paired up new index added to it .
# eg from above the id of char 't' is 116 and 'h' is 104
# so pair of (116,104) -> 'th' occurs 29 times in the above initial tokens produced by the text_corous we are using.
# so when producing the new tokens list we replace (116,104) with 256 and out lookup table then means that word -> 'th' has the id 256

def pair_up(input_tokens_ids,pair,new_index):
    new_dictionary = []
    i=0
    while i<len(input_tokens_ids):
        if i<len(input_tokens_ids)-2 and input_tokens_ids[i] == pair[0] and input_tokens_ids[i+1] == pair[1]:
            new_dictionary.append(new_index)
            i+=2
        else:
            new_dictionary.append(input_tokens_ids[i])
            i+=1
    return new_dictionary

# new_list = pair_up(tokens,(116,104),256) # 1334-29 = 1305
# new_list = pair_up(new_list,(101,32),257)  # 1305-24 = 1281

# print(len(new_list))

In [2]:
#  Manual compression follow below cell for updated code........

new_list = tokens
start_id = 256
for i,j in sorted_stats:
    if i>=2:
        new_list = pair_up(new_list,j,start_id)
        start_id+=1
# print(new_list)
logger("length of new list "+str(len(new_list)))
stats_2 = get_stats(new_list)
repositioned_stats_2 = tuple((v,k) for k,v in stats_2.items()) 
sorted_stats_2 = sorted(repositioned_stats_2,reverse=True)
# print(sorted_stats_2)
new_list_2 = new_list
for i,j in sorted_stats_2:
    if i>=2:
        new_list_2 = pair_up(new_list_2,j,start_id)
        start_id+=1
logger("length of new list "+str(len(new_list_2)))
stats_3 = get_stats(new_list_2)
repositioned_stats_3 = tuple((v,k) for k,v in stats_3.items()) 
sorted_stats_3 = sorted(repositioned_stats_3,reverse=True)
print(sorted_stats_3)

NameError: name 'tokens' is not defined

In [33]:
import json
vocab_size = 276 # Hyper param
max_merges = vocab_size - 256
merges = {} # this dict will be storing the (id,id) -> id : eg : is 116 and 104 are combined to 256 then -> (116,104) -> 256 , this help
            # us in decoding and etc..

ids = tokens
for i in range(max_merges):
    stats = get_stats(ids)
    top_pair = max(stats,key=stats.get)
    indx = 256+i
    logger(f"merging pairs {top_pair} in to new token id  : {indx}")
    ids = pair_up(ids,top_pair,indx)
    merges[top_pair]= indx
# merges = {(101, 32): 256, (115, 32): 257, (116, 104): 258, (110, 32): 259, (100, 32): 260, (101, 114): 261, (97, 110): 262, (116, 32): 263, (258, 256): 264, (44, 32): 265, (105, 110): 266, (111, 114): 267, (97, 114): 268, (46, 32): 269, (121, 32): 270, (97, 108): 271, (101, 260): 272, (10, 10): 273, (111, 102): 274, (111, 110): 275}
str_merges = {f"{k[0]},{k[1]}":v for k,v in merges.items()}
with open("vocab.json","w") as f:
    json.dump(str_merges, f, indent=4)

In [14]:
logger(f"Compression : {len(tokens)/len(ids):.2f}")
logger(f"Number of ids : {len(ids)}")

************************************************************
message --> Compression : 1.27
************************************************************
************************************************************
message --> Number of ids : 140441631
************************************************************


In [34]:
import json
def load_merges_dict(fileName:str):
        with open(fileName, "r",encoding="utf-8") as f:
            loaded_data = json.load(f)
        res = {tuple(map(int, k.split(","))): v for k, v in loaded_data.items()}
        return res
merges = load_merges_dict("merge_2.json")
print(merges)

{(101, 32): 256, (115, 32): 257, (116, 104): 258, (110, 32): 259, (100, 32): 260, (101, 114): 261, (97, 110): 262, (116, 32): 263, (258, 256): 264, (44, 32): 265, (105, 110): 266, (111, 114): 267, (97, 114): 268, (46, 32): 269, (121, 32): 270, (97, 108): 271, (101, 260): 272, (10, 10): 273, (111, 102): 274, (111, 110): 275}


In [35]:
# This Function take in the sequence of the token_ids:list and produces a python string object eg : [65,66,67] -> ABC 
#  using the  decode method with (utf-8) decoding .

vocab = {indx : bytes([indx]) for indx in range(256)}
for pair,_id in merges.items():
    vocab[_id]=vocab[pair[0]]+vocab[pair[1]]
    
def decoder(token_ids):
    tks = b"".join(vocab[i] for i in token_ids)
    final_text = tks.decode("utf-8",errors="replace")
    return final_text    

In [41]:
decoder([104, 101, 108, 108, 111, 32, 119, 267, 108, 100, 33])

'hello world!'

In [37]:
def encoder(text_seq):
    
    merges = load_merges_dict("merge.json")
    _tokens = list(map(int,text_seq.encode("utf-8")))
    while len(_tokens)>=2 :
        base_stats = get_stats(_tokens)
        min_pair = min(base_stats,key=lambda p:merges.get(p,float("inf")))
        if min_pair not in merges:
            break
        index = merges.get(min_pair)
        _tokens = pair_up(_tokens,min_pair,index)
    return _tokens



In [40]:
encoder("hello world!")

[104, 101, 108, 108, 111, 32, 119, 267, 108, 100, 33]